# Challenge 6 — Estadística para Data Science

__Nombre:__ Abigail Chirinos  
__Clase:__ Estadística para Data Science

__Objetivo:__ Utilizar inferencia estaística para evaluar si una diferencia obsevada puede considerarse estadística significativa

En este ejercicio se utilizan:

- `dataset_limpio.csv`

__Challenge__
Dos grupos presenta resultados distintos.
Tu tarea es determinar si existe evidencia estadística suficiente para afirmar que susmedias sondiferentes.


__Pregunta__

**¿Existe una diferencia estadísticamente significativa entre el total promedio de las ventas de Deportes y Ropa?**

Para el ejercicio se comparan dos grupos del dataset: `Deportes` y `Ropa`.


## 1. Cargar los datos

In [6]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, t

df = pd.read_csv("dataset_limpio.csv")

df.head()

,fecha,region_id,vendedor,producto_id,cantidad,nombre_producto,categoria,precio_unitario,total
0,2023-01-01,1,Vendedor_4,7,9,Producto 7,Electrónica,1932.73,17394.57
1,2023-01-01,3,Vendedor_10,5,5,Producto 5,Ropa,1240.04,6200.20
2,2023-01-01,5,Vendedor_6,1,1,Producto 1,Electrónica,1264.57,1264.57
3,2023-01-01,4,Vendedor_9,14,3,Producto 14,Deportes,2949.45,8848.35
4,2023-01-01,2,Vendedor_6,17,4,Producto 17,Ropa,1593.14,6372.56


## Separar los dos grupos

Se utilizará la variable `total` como resultado que vamos a comparar.

- Grupo A: Deportes
- Grupo B: Ropa


In [8]:
grupo_a = df.loc[df["categoria"] == "Deportes", "total"]
grupo_b = df.loc[df["categoria"] == "Ropa", "total"]

print("Deportes:", len(grupo_a))
print("Ropa:", len(grupo_b))


Deportes: 2118
Ropa: 1555


## Paso 1 Estadística descriptiva

In [9]:
resumen = pd.DataFrame({
    "n": [len(grupo_a), len(grupo_b)],
    "media": [grupo_a.mean(), grupo_b.mean()],
    "mediana": [grupo_a.median(), grupo_b.median()],
    "desviacion_std": [grupo_a.std(), grupo_b.std()]
}, index=["Deportes", "Ropa"])

# Error estándar
resumen["error_std"] = resumen["desviacion_std"] / np.sqrt(resumen["n"])

# Intervalo de confianza del 95%
resumen["IC_95_inf"] = resumen["media"] - 1.96 * resumen["error_std"]
resumen["IC_95_sup"] = resumen["media"] + 1.96 * resumen["error_std"]

resumen

,n,media,mediana,desviacion_std,error_std,IC_95_inf,IC_95_sup
Deportes,2118,12115.794169,9855.88,9561.041001,207.750560,11708.603072,12522.985266
Ropa,1555,11927.848707,11129.22,6983.324618,177.091222,11580.749913,12274.947502


#### Intervalos de confianza del 95%

In [10]:
def intervalo_confianza(grupo):
    media = grupo.mean()
    error = grupo.std() / np.sqrt(len(grupo))
    margen = t.ppf(0.975, len(grupo) - 1) * error
    return media - margen, media + margen
ic_deportes = intervalo_confianza(grupo_a)
ic_ropa = intervalo_confianza(grupo_b)
print("IC 95% Deportes:", ic_deportes)
print("IC 95% Ropa:", ic_ropa)

IC 95% Deportes: (np.float64(11708.377621686355), np.float64(12523.210716368412))
IC 95% Ropa: (np.float64(11580.485743839594), np.float64(12275.211670951407))


## Paso2 - Hipótesis

**H0:** μ Deportes = μ Ropa

**H1:** μ Deportes ≠ μ Ropa

Usamos:

**α = 0.05**

La prueba que se utilizará es una **t de Welch**, porque permite comparar las medias de dos grupos sin asumir que sus varianzas son iguales.


In [11]:
t_stat, p_val = ttest_ind(grupo_a,grupo_b,equal_var=False)

print("t:", t_stat)
print("p-value:", p_val)

t: 0.6884791138605041
p-value: 0.49119462410396547


### Paso 4 - Decisión

Si `p < 0.05`, se rechaza H0.

Si `p >= 0.05`, no se rechaza H0.


In [12]:
alpha = 0.05

if p_val < alpha:
    print("Se rechaza H0.")
    print("Hay evidencia estadística de una diferencia entre las medias.")
else:
    print("No se rechaza H0.")
    print("No hay evidencia estadística suficiente para afirmar una diferencia entre las medias.")


No se rechaza H0.
No hay evidencia estadística suficiente para afirmar una diferencia entre las medias.


## Paso 5 -- Magnintud del efecto

Calcular o investigar el tamaño del efecto 
Responder:  
> - ¿Una diferencia estaística significativa también es importante en términos práctivos?

También revisamos la diferencia con Cohen's d para saber si la diferencia observada es pequeña, mediana o grande en términos relativos.

In [15]:
media_a = grupo_a.mean()
media_b = grupo_b.mean()

sd_pooled = np.sqrt(((len(grupo_a)-1) * grupo_a.var(ddof=1) + (len(grupo_b)-1) * grupo_b.var(ddof=1)) / (len(grupo_a) + len(grupo_b) - 2))
cohen_d = (media_a - media_b) / sd_pooled

print("Cohen's d:", cohen_d)

Cohen's d: 0.021943252189046605


El tamaño del efecto complementa al valor p. Una diferencia puede ser estadísticamente significativa y aun así ser pequeña en términos prácticos, por lo que conviene mirar ambas cosas.


## Paso 6 - Correlación

In [16]:
correlacion = df[["cantidad", "total"]].corr().loc["cantidad", "total"]

print("Correlación entre cantidad y total:", correlacion)


Correlación entre cantidad y total: 0.6683195126836309


La correlación de Pearson mide la dirección y fuerza de la relación lineal entre `cantidad` y `total`.

Una correlación positiva indica que, en general, cuando aumenta la cantidad, también tiende a aumentar el total. Esto **no demuestra causalidad**.


## Preguntas de reflexión

1. **¿Existe diferencia estadísticamente significativa?**
   No encontré evidencia estadística suficiente para afirmar que las medias de Deportes y Ropa sean diferentes. La prueba t de Welch obtuvo un p-value de 0.4912, mayor que 0.05, por lo que no rechacé la hipótesis nula. Esto no demuestra que las medias sean iguales.

2. **¿Qué indica el valor p?**
   El valor p indica qué tan compatibles son los resultados con la hipótesis de que ambas medias son iguales. Si esta hipótesis y los supuestos de la prueba fueran correctos, la probabilidad de obtener un resultado tan extremo como el observado, o más, sería aproximadamente del 49.12 %. Por eso, no considero que la diferencia observada sea evidencia suficiente para rechazar la hipótesis nula.

3. **¿Los intervalos de confianza son amplios o estrechos?**
   Considero que son relativamente estrechos respecto al valor de las medias. Utilizando la distribución t, el intervalo del 95 % para Deportes va de 11,708.38 a 12,523.21, y para Ropa de 11,580.49 a 12,275.21. Ambos se superponen ampliamente, aunque la decisión sobre la diferencia entre medias la basé en la prueba de Welch.

4. **¿El tamaño del efecto parece importante?**
   El tamaño del efecto fue muy pequeño: obtuve un Cohen’s d de 0.0219. Aunque Deportes presentó una media aproximadamente 187.95 unidades monetarias mayor que Ropa, esta diferencia es pequeña respecto a la variabilidad de las ventas. Una diferencia estadísticamente significativa tampoco sería necesariamente importante en términos prácticos; tendría que considerar su magnitud y el contexto del negocio.

5. **¿Qué significa la correlación encontrada?**
   Encontré una correlación positiva de 0.6683 entre `cantidad` y `total`. Esto indica una relación lineal moderadamente fuerte: las ventas con mayor cantidad tienden a tener un importe total más alto. Además, esta relación es esperable porque la cantidad interviene en el cálculo del total. La correlación por sí sola no demuestra causalidad.

## Conclusión

Al comparar las categorías Deportes y Ropa, observé que Deportes tuvo un importe promedio por venta ligeramente mayor: 12,115.79 frente a 11,927.85. Sin embargo, la prueba estadística no mostró evidencia suficiente para afirmar que sus medias sean diferentes, y el tamaño del efecto fue muy pequeño. Por ello, con este análisis no puedo concluir que una categoría tenga un importe promedio por venta superior a la otra. También encontré que una mayor cantidad de productos vendidos se relaciona con un mayor importe total.
